# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer: Data Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities, such as record sets, fields, and columns, are referenced by their `@id` as per best practice for Croissant datasets.

### Dataset Source
The dataset is described and structured by a Croissant schema accessible at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install `mlcroissant` if not available
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and tabular records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset Croissant package
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object (not as dict/list)
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {md.datePublished}\nLicense: {md.license}")
print(f"Cite as: {md.citeAs}")

## 2. Data Overview
Let's examine which Record Sets are available in this dataset, and what fields (columns) each contains. All identifiers (`@id`) are shown for consistency.

We use the metadata model to get all available record sets by their `@id`, and for each record set, enumerate its fields (also by `@id`).

In [ ]:
# List all Record Sets with their @id and field @ids

record_sets = []
print('Available Record Sets and Fields:')
print('-'*60)
if hasattr(md, 'recordSet') and md.recordSet:
    rs_list = md.recordSet
    # Single recordset if only one
    if not isinstance(rs_list, list):
        rs_list = [rs_list]
    for rs in rs_list:
        print(f"Record Set @id: {rs['@id']} | name: {rs.get('name','')}")
        record_sets.append(rs['@id'])
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for f in fields:
            # f is a dict, show @id and name if available
            if isinstance(f, dict):
                print(f"    Field @id: {f['@id']}, name: {f.get('name','')}")
            else:
                print(f"    Field @id: {f}")
else:
    # If not defined in metadata, query via dataset API (fallback)
    for rs in dataset.available_record_sets():
        print(f"Record Set @id: {rs['@id']} | name: {rs.get('name','')}")
        record_sets.append(rs['@id'])
        for f in rs.get('field', []):
            # Field may be ID or dict
            if isinstance(f, dict):
                print(f"    Field @id: {f['@id']}, name: {f.get('name','')}")
            else:
                print(f"    Field @id: {f}")
if not record_sets:
    # Try to find record set IDs using dataset.available_record_sets (mlcroissant >=1.1)
    rs_ids = dataset.available_record_sets()
    for rs in rs_ids:
        print(f"Record Set @id: {rs}")
        record_sets.append(rs)
    
print(f"\nRecord sets discovered: {record_sets}")

## 3. Data Extraction
We will extract all rows from the primary record set, identified by its `@id`. All fields/columns will be referenced by their `@id`.

**Note:** Replace `<recordset_id>` with the appropriate `@id` for your main tabular dataset based on the output above.
For the FAIR2 CRC dataset, the main clinical data is typically in the first record set.

In [ ]:
# If you found multiple record sets, edit this list to include those @id strings. Example:
# record_sets = ['cr:RecordSet_crc_patients']
if not record_sets:
    # Try commonly used default for Croissant
    primary_recordset = 'cr:RecordSet_1'
else:
    primary_recordset = record_sets[0]  # Use first discovered or adjust as needed

# Extract data into DataFrames for each record set
dataframes = {}
for rsid in record_sets:
    print(f"Loading records for Record Set @id: {rsid}")
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
    else:
        print(f"No records found for {rsid}")

# List columns (these are @id of fields in the schema)
if primary_recordset in dataframes:
    df = dataframes[primary_recordset]
    print(f"Columns (field @id) for {primary_recordset}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print(f"Primary recordset {primary_recordset} not found in dataframes.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic analysis: filter for numeric fields, remove outliers, normalize values, and group the data. All column access is by field `@id`.

Adjust the code below by choosing one or more **numeric field `@id`** and one **group field `@id`** from your DataFrame above. Typical numeric fields might represent age, interval (months), or counts. Grouping might be by sex, anatomical location, or other categorical columns.

In [ ]:
# Choose a numeric field and a group field by @id
# Replace these with actual @ids from the DataFrame
numeric_field_id = None
group_field_id = None

print("Available columns (field @id):")
print(df.columns.tolist())

# Auto-search for likely numeric fields (e.g. containing 'age', 'interval', 'count', etc.)
possible_numeric = [col for col in df.columns if ('age' in col.lower() or 'interval' in col.lower() or 'month' in col.lower() or df[col].dtype in [np.int64, np.float64])]
if possible_numeric:
    numeric_field_id = possible_numeric[0]
    print(f"Guessing numeric field: {numeric_field_id}")
else:
    print("No obvious numeric field detected. Please set numeric_field_id manually.")

# Try sex or location for grouping
for gcol in df.columns:
    if any(x in gcol.lower() for x in ['sex', 'gender', 'anatomical', 'location']):
        group_field_id = gcol
        print(f"Guessed group field: {group_field_id}")
        break
if not group_field_id:
    print("No obvious group field detected. Please set group_field_id manually if needed.")

if numeric_field_id:
    # Show value counts
    col = numeric_field_id
    print(f"Summary statistics for numeric field '{col}':")
    print(df[col].describe())

    # Define a threshold for filtering (e.g., above mean)
    mean = df[col].mean()
    threshold = mean
    filtered_df = df[df[col] > threshold]
    print(f"\nFiltered records with {col} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
    print(f"\nNormalized {col} for filtered records:")
    display(filtered_df[[col, f"{col}_normalized"]].head())

    # Group by group_field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[col].mean().to_frame(f"mean_{col}")
        print(f"\nGrouped average of {col} by {group_field_id}:")
        display(grouped_df)
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if available, mean by group. All axes labels use the corresponding field `@id`.

If `matplotlib` or `seaborn` is not installed, run `!pip install matplotlib seaborn` first in a cell.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped data available
    if group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization skipped: No numeric field available.")

## 6. Conclusion
This notebook demonstrated step-by-step exploration of the FAIR^2 dataset using the `mlcroissant` library, including programmatic reference by `@id` for all Croissant entities. We reviewed available record sets and fields, loaded the primary tabular data, performed basic filtering and normalization, and visualized key field(s).

**Key next steps:**
- Review the definitions and descriptions of each field in the Croissant schema for context.
- Consider more detailed statistical or predictive analyses on the cleaned dataset.
- Share your analysis code using the `@id` references for reproducibility and clarity.

For further documentation see: https://github.com/mlcommons/croissant
